# Advanced-MRI-Breast-Lesions preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.advanced_mri_lesions import AdvancedMRILesions, PATHOLOGY_TO_MRI_TYPE, csv_save_path
from utils.preprocessing import UNKNOWN, birads_mapping

ds = AdvancedMRILesions()
ds.set_dataset_name("advanced-mri-lesions")
print(ds.mri_lesions_df.shape, ds.metadata_df.shape)
ds.mri_lesions_df.head()

### Step 1 — drop pathology/receptor columns not used by the common schema

In [ ]:
ds.mri_lesions_df = ds.mri_lesions_df.drop(columns=[
    "tumor/benign1", "GRADE1", "ER [SII] 1", "PR [SII] 1", "HER2 [SII] 1", "isTN1", "ER [%] 1",
    "PR [%] 1", "HER2 [%] 1", "Unnamed: 17", "KI67[%] 1",
    "tumor/benign2", "GRADE2", "ER [SII] 2", "PR [SII] 2", "HER2 [SII] 2", "isTN2", "ER [%] 2",
    "PR  [%] 2", "HER  [%] 2", "Unnamed: 30", "KI67[%] 2",
    "tumor/benign3", "GRADE3", "ER [SII] 3", "PR [SII] 3", "HER [SII] 3", "isTN3", "ER [%] 3",
    "PR  [%] 3", "HER  [%] 3", "Unnamed: 43", "KI67[%] 3",
    "tumor/benign4", "tumor/benign5", "tumor/benign6",
])
ds.mri_lesions_df.shape

### Step 2 — treat the sheet's `-1` / `-1.0` sentinel values as missing

In [ ]:
ds.mri_lesions_df = ds.mri_lesions_df.replace(-1, None)
ds.mri_lesions_df = ds.mri_lesions_df.replace(-1.0, None)
ds.mri_lesions_df.isna().sum().head()

### Step 3 — clean column names and align the patient id column with the metadata sheet

In [ ]:
from utils.preprocessing import csv_column_cleaning

ds.mri_lesions_df.columns = [col.replace("id#", "") for col in csv_column_cleaning(list(ds.mri_lesions_df.columns))]
ds.mri_lesions_df = ds.mri_lesions_df.rename(columns={"patient id": "subject id"})
ds.mri_lesions_df.columns.tolist()

### Step 4 — classify each of the up-to-6 lesions per patient into mass vs. unsupported finding types

In [ ]:
for i in range(1, 7):
    ds.mri_lesions_df[f"type lesion{i}"] = ds.mri_lesions_df[f"pathology{i}"].apply(
        lambda x: (PATHOLOGY_TO_MRI_TYPE.get(int(x), None) if pd.notna(x) else None)
    )
ds.mri_lesions_df["type lesion1"].value_counts(dropna=False)

### Step 5/6 — clean the TCIA metadata sheet and keep only the columns needed to locate files

In [ ]:
adjusted_columns = list(ds.metadata_df.columns[1:])
adjusted_columns.insert(adjusted_columns.index("file size"), UNKNOWN)
ds.metadata_df.columns = adjusted_columns

meta_cols_to_keep = ["subject id", "study date", "study description", "series description",
                     "file location", "modality", "manufacturer"]
ds.metadata_df = ds.metadata_df[meta_cols_to_keep]
ds.metadata_df.head()

### Step 7 — split metadata into the MR series and the SEG (segmentation) series

In [ ]:
from copy import deepcopy

seg_metadata_df = deepcopy(ds.metadata_df[ds.metadata_df["modality"] == "SEG"]).reset_index(drop=True)
mr_metadata_df = deepcopy(ds.metadata_df[ds.metadata_df["modality"] == "MR"]).reset_index(drop=True)
len(seg_metadata_df), len(mr_metadata_df)

### Step 8 — resolve each SEG row's `file location` to an actual file path on disk

In [ ]:
import glob
import os

from preprocessing.advanced_mri_lesions import raw_imgs_path

seg_metadata_df["segmentation path"] = seg_metadata_df["file location"].apply(
    lambda x: glob.glob(os.path.join(raw_imgs_path, x[2:], "*"), recursive=True)[0]
)
seg_metadata_df = seg_metadata_df[["subject id", "segmentation path"]]
seg_metadata_df.head()

### Step 9 — join the lesion sheet with the MR series (inner) and the segmentation path (left)

In [ ]:
ds.mri_lesions_df = pd.merge(ds.mri_lesions_df, mr_metadata_df, on=["subject id"], how="inner")
ds.mri_lesions_df = pd.merge(ds.mri_lesions_df, seg_metadata_df, on=["subject id"], how="left")
ds.mri_lesions_df.shape

### Step 10 — drop segmentation paths whose series doesn't match the expected sequence
Only `Registered AX Sen Vibrant MultiPhase` was manually validated to align with the segmentation.

In [ ]:
ds.mri_lesions_df["segmentation path"] = ds.mri_lesions_df.apply(
    lambda x: ds.remove_non_mri_segmentation_paths(x), axis=1
)
ds.mri_lesions_df["segmentation path"].notna().sum()

### Step 11 — fill in derived clinical content (referral reasons, implants, BI-RADS, lesion side/slice positions)

In [ ]:
ds.fill_in_contents()
ds.mri_lesions_df[["reason for referral ", "breast implants", "birads"]].head()

### Step 12/13/14 — keep only segmented, BI-RADS-known, non-additional-evaluation rows

In [ ]:
ds.mri_lesions_df = ds.mri_lesions_df[ds.mri_lesions_df["segmentation path"].notnull()].reset_index(drop=True)
ds.mri_lesions_df = ds.mri_lesions_df[ds.mri_lesions_df["birads"].notnull()].reset_index(drop=True)
ds.mri_lesions_df = ds.mri_lesions_df[ds.mri_lesions_df["birads"] != birads_mapping[0]].reset_index(drop=True)
ds.mri_lesions_df.shape

### Step 15/16 — fold known-biopsy-proven (6) into highly-suggestive (5), cast age to int

In [ ]:
from utils.preprocessing import sanitize_age

ds.mri_lesions_df["birads"] = ds.mri_lesions_df["birads"].replace({birads_mapping[6]: birads_mapping[5]})
ds.mri_lesions_df["age at mri"] = ds.mri_lesions_df["age at mri"].apply(lambda x: sanitize_age(x))
ds.mri_lesions_df["birads"].value_counts(dropna=False)

## Build exam records for one patient

In [ ]:
row = ds.mri_lesions_df.iloc[0]
exams = ds.process_row(row)
len(exams), exams[0] if exams else None

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"lesion rows before per-slice processing: {len(ds.mri_lesions_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))